In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()

model = ChatOpenAI(model='gpt4-o-mini')

True

In [5]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detail feedback')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [6]:
essay = """"""

In [ ]:
prompt = f'Evaluate the language quality of the followning essay and provide the feedback and assign the score out of 10 \n{essay}'

structured_model.invoke(prompt)

# to find score
structured_model.invoke(prompt).score


In [ ]:
# Creating the workflow
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

graph = StateGraph(EssayState)

def evaluate_language(state: EssayState):
    prompt = f'Evaluate the language quality of the followning essay and\
        provide the feedback and assign the score out of 10 \n{state['essay']}'
    output = structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

def evaluate_analysis(state: EssayState):
    prompt = f'Evaluate the depth of analysis of the followning essay and\
        provide the feedback and assign the score out of 10 \n{state['essay']}'
    output = structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}


def evaluate_thought(state: EssayState):
    prompt = f'Evaluate the clarity of thought of the followning essay and\
        provide the feedback and assign the score out of 10 \n{state['essay']}'
    output = structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

def final_evaluation(state: EssayState):
    # Summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    output = model.invoke(prompt).content

    # average calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': output, 'avg_score': avg_score}


# Adding nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
initial_state = {
    'essay': essay
}
workflow.invoke(initial_state)
